In [32]:
import math

In [33]:
GTX_1070 = {
    # Chip
    'arch': 'Pascal GP104',
    'compute_capability': 6.1,
    'num_SMs': 15,
    'cores_per_SM': 128,           # FP32 CUDA cores
    'total_cores': 15 * 128,       # 1920
    'boost_clock_MHz': 1683,

    # Per-SM resources
    'registers_per_SM': 65536,     # 32-bit registers
    'shared_mem_per_SM_KB': 96,
    'L1_cache_per_SM_KB': 24,      # separate from shared on Pascal
    'constant_cache_per_SM_KB': 8, # approximate

    # Per-SM residency limits
    'max_threads_per_SM': 2048,
    'max_warps_per_SM': 64,
    'max_TBs_per_SM': 32,
    'warp_schedulers_per_SM': 4,

    # Per-TB limits
    'max_threads_per_TB': 1024,
    'max_shared_mem_per_TB_KB': 48,
    'max_registers_per_thread': 255,

    # Per-thread
    'warp_size': 32,

    # Memory system
    'device_memory_GB': 8,
    'memory_bandwidth_GBps': 256,
    'L2_cache_MB': 2,

    # Execution latencies (cycles)
    'FMA_latency_cycles': 6,
    'shared_mem_latency_cycles': 30,
    'global_mem_latency_cycles': 500,   # approximate, varies

    # Bank organization
    'shared_mem_banks': 32,
    'bank_width_bytes': 4,
}

In [ ]:
RTX_3060 = {
    # Chip
    'arch': 'Ampere GA106',
    'compute_capability': 8.6,
    'num_SMs': 28,
    'cores_per_SM': 128,           # 64 dedicated FP32 + 64 FP32/INT32 dual-use
    'total_cores': 28 * 128,       # 3584
    'boost_clock_MHz': 1777,

    # Per-SM resources
    'registers_per_SM': 65536,     # 32-bit registers (same as Pascal)
    'shared_mem_per_SM_KB': 100,   # configurable L1/shared carveout, up to 100 KB
    'L1_cache_per_SM_KB': 28,      # unified with shared; 128 KB total minus shared carveout
    'constant_cache_per_SM_KB': 8, # approximate

    # Per-SM residency limits
    'max_threads_per_SM': 1536,    # down from 2048 on Pascal
    'max_warps_per_SM': 48,        # down from 64 on Pascal
    'max_TBs_per_SM': 16,          # down from 32 on Pascal
    'warp_schedulers_per_SM': 4,

    # Per-TB limits
    'max_threads_per_TB': 1024,
    'max_shared_mem_per_TB_KB': 48,  # static; 99 KB with dynamic opt-in
    'max_registers_per_thread': 255,

    # Per-thread
    'warp_size': 32,

    # Memory system
    'device_memory_GB': 12,
    'memory_bandwidth_GBps': 360,
    'L2_cache_MB': 3,              # approximate

    # Execution latencies (cycles)
    'FMA_latency_cycles': 4,       # down from 6 on Pascal
    'shared_mem_latency_cycles': 19,  # down from 30 on Pascal
 
    # Bank organization
    'shared_mem_banks': 32,
    'bank_width_bytes': 4,
}

In [35]:
def multi_block_size(N_T, N_reg):
    return N_T, N_reg

In [36]:
def compute_occupancy(N_T, N_reg, batch_size, specs=RTX_3060):
    """
    Compute achievable GPU occupancy for a multi-block IIR filtering kernel.
    
    In this model, N_reg equals N (the number of blocks per signal block group):
    each thread holds N register-resident sample values, one per register.
    Overhead registers (coefficients, temporaries, indices) are treated as
    absorbed into a lower-order correction and ignored.
    
    Parameters
    ----------
    N_T : int
        Number of threads per TB (= block size L for PH/STCR).
    N_reg : int
        Number of registers per thread (= N, number of samples per thread).
    batch_size : int
        Total number of samples S in the batch.
    specs : dict
        GPU spec dictionary (default: RTX_3060).
    
    Returns
    -------
    dict with occupancy metrics.
    """
    N = N_reg  # by construction in this model
    notes = []
    
    threads_per_TB = N_T
    warps_per_TB = -(-threads_per_TB // specs['warp_size'])  # ceiling div
    
    # Shared memory for the stride-N permutation (with +1 padding per row)
    smem_per_TB_bytes = N_T * (N + 1) * 4
    smem_per_TB_KB = smem_per_TB_bytes / 1024
    
    # Per-TB register footprint
    regs_per_TB = N_reg * threads_per_TB
    
    # --- Hard constraint checks ---
    if N_reg > specs['max_registers_per_thread']:
        notes.append(
            f"N_reg={N_reg} exceeds per-thread cap "
            f"{specs['max_registers_per_thread']}; compiler will spill."
        )
    if threads_per_TB > specs['max_threads_per_TB']:
        notes.append(
            f"threads_per_TB={threads_per_TB} exceeds per-TB cap "
            f"{specs['max_threads_per_TB']}; kernel will not launch."
        )
    if smem_per_TB_KB > specs['max_shared_mem_per_TB_KB']:
        notes.append(
            f"smem_per_TB={smem_per_TB_KB:.1f} KB exceeds per-TB cap "
            f"{specs['max_shared_mem_per_TB_KB']} KB; kernel will not launch."
        )
    
    # --- Resource-driven TB residency limits ---
    tb_from_regs = specs['registers_per_SM'] // regs_per_TB \
                   if regs_per_TB > 0 else specs['max_TBs_per_SM']
    tb_from_smem = int((specs['shared_mem_per_SM_KB'] * 1024)
                       // smem_per_TB_bytes) \
                   if smem_per_TB_bytes > 0 else specs['max_TBs_per_SM']
    tb_from_threads = specs['max_threads_per_SM'] // threads_per_TB
    tb_from_warps = specs['max_warps_per_SM'] // warps_per_TB
    tb_from_hw = specs['max_TBs_per_SM']
    
    tbs_supply = min(tb_from_regs, tb_from_smem, tb_from_threads,
                     tb_from_warps, tb_from_hw)
    
    binding = min(
        [('registers', tb_from_regs),
         ('shared_memory', tb_from_smem),
         ('threads', tb_from_threads),
         ('warps', tb_from_warps),
         ('hardware_TB_cap', tb_from_hw)],
        key=lambda x: x[1]
    )[0]
    
    # --- Batch-driven demand ---
    samples_per_TB = N * N_T
    total_TBs_in_batch = batch_size / samples_per_TB
    tbs_demand_per_SM = total_TBs_in_batch / specs['num_SMs']
    
    tbs_effective = min(tbs_supply, tbs_demand_per_SM)
    warps_per_SM = tbs_effective * warps_per_TB
    
    # --- Volkov saturation check ---
    volkov = (specs['FMA_latency_cycles']
              * specs['warp_schedulers_per_SM'] / 3)  # ILP=3 assumption
    
    if total_TBs_in_batch < specs['num_SMs']:
        notes.append(
            f"total_TBs_in_batch={total_TBs_in_batch:.1f} < num_SMs="
            f"{specs['num_SMs']}; some SMs will be idle."
        )
    if batch_size % samples_per_TB != 0:
        notes.append(
            f"batch_size={batch_size} is not a multiple of samples_per_TB="
            f"{samples_per_TB}; fractional TBs shown."
        )
    
    return {
        'N_T': N_T,
        'N_reg': N_reg,
        'batch_size': batch_size,
        'samples_per_TB': samples_per_TB,
        'threads_per_TB': threads_per_TB,
        'warps_per_TB': warps_per_TB,
        'smem_per_TB_bytes': smem_per_TB_bytes,
        'smem_per_TB_KB': round(smem_per_TB_KB, 2),
        'regs_per_TB': regs_per_TB,
        'total_TBs_in_batch': total_TBs_in_batch,
        'max_TBs_per_SM_from_registers': tb_from_regs,
        'max_TBs_per_SM_from_smem': tb_from_smem,
        'max_TBs_per_SM_from_threads': tb_from_threads,
        'max_TBs_per_SM_from_warps': tb_from_warps,
        'max_TBs_per_SM_from_hw_cap': tb_from_hw,
        'binding_resource': binding,
        'TBs_per_SM_supply': tbs_supply,
        'TBs_per_SM_demand': round(tbs_demand_per_SM, 2),
        'TBs_per_SM_effective': round(tbs_effective, 2),
        'warps_per_SM': round(warps_per_SM, 2),
        'volkov_threshold': volkov,
        'meets_volkov': warps_per_SM >= volkov,
        'occupancy_ratio': round(warps_per_SM / volkov, 3),
        'notes': notes,
    }


def print_occupancy(result, specs=RTX_3060):
    """Pretty-print an occupancy result with hardware limits shown alongside."""
    
    label_w = 33
    val_w = 20
    
    def line(label, value, limit_label=None, limit_value=None):
        left = f"  {label:<{label_w}} {str(value):<{val_w}}"
        if limit_label is not None:
            right = f"| {limit_label}: {limit_value}"
            print(left + right)
        else:
            print(left)
    
    print(f"Configuration: N_T={result['N_T']}, N_reg={result['N_reg']}, "
          f"batch_size={result['batch_size']:,}")
    print("-" * 95)
    
    # -------- Per-TB footprint vs. per-TB hardware caps --------
    line("Threads per TB:",
         result['threads_per_TB'],
         "max threads per TB",
         specs['max_threads_per_TB'])
    line("Warps per TB:",
         result['warps_per_TB'])
    line("Registers per thread (N_reg):",
         result['N_reg'],
         "max registers per thread",
         specs['max_registers_per_thread'])
    line("Registers per TB:",
         result['regs_per_TB'])
    line("Shared memory per TB:",
         f"{result['smem_per_TB_KB']} KB",
         "max shared mem per TB",
         f"{specs['max_shared_mem_per_TB_KB']} KB")
    line("Samples per TB (N_reg * N_T):",
         result['samples_per_TB'])
    
    print()
    
    # -------- Batch-level info --------
    line("Batch size (S):",
         f"{result['batch_size']:,}",
         "device memory",
         f"{specs['device_memory_GB']} GB")
    line("Total TBs in batch:",
         f"{result['total_TBs_in_batch']:.1f}",
         "num SMs",
         specs['num_SMs'])

    print()
    
    # -------- Per-SM utilization at effective residency --------
    tbs_eff = result['TBs_per_SM_effective']
    
    regs_used = result['regs_per_TB'] * tbs_eff
    smem_used_KB = result['smem_per_TB_KB'] * tbs_eff
    threads_used = result['threads_per_TB'] * tbs_eff
    warps_used = result['warps_per_TB'] * tbs_eff
    
    regs_max = specs['registers_per_SM']
    smem_max = specs['shared_mem_per_SM_KB']
    threads_max = specs['max_threads_per_SM']
    warps_max = specs['max_warps_per_SM']
    tbs_max = specs['max_TBs_per_SM']
    
    print(f"  Per-SM utilization at effective residency (TBs/SM = {tbs_eff}):")
    line("    registers used per SM:",
         f"{regs_used:,.0f}",
         "max registers per SM",
         f"{regs_max}")
    line("    shared memory used per SM:",
         f"{smem_used_KB:.2f}",
         "max shared mem per SM",
         f"{smem_max} KB")
    line("    threads used per SM:",
         f"{threads_used:,.0f}",
         "max registers per thread",
         f"{threads_max}")
    
    print()
    
    # -------- Latency-hiding check --------
    line("TBs used per SM:",
         f"{tbs_eff} ",
         "max TBs per SM",
         f"{tbs_max}")
    line("Warps/SM (effective):",
         result['warps_per_SM'],
         "max warps per SM",
         warps_max)
    line("Volkov threshold (~5.3, ILP=3):",
         f"{result['volkov_threshold']:.1f} warps",
         "FMA latency x schedulers",
         f"{specs['FMA_latency_cycles']} x {specs['warp_schedulers_per_SM']}")
    line("Meets Volkov?",
         "YES" if result['meets_volkov'] else "NO — below threshold")
    line("Occupancy ratio:",
         f"{result['occupancy_ratio']:.1%}")
    waves = math.ceil(result['total_TBs_in_batch'] / (result['TBs_per_SM_supply'] * specs['num_SMs']))
    line("Waves:", f"{waves:d}")
    
    # -------- Notes --------
    if result['notes']:
        print()
        print(f"  Notes:")
        for note in result['notes']:
            print(f"    - {note}")
    print()

In [37]:
N_T, N_reg = multi_block_size(N_T=32, N_reg=32)

In [38]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**16))

Configuration: N_T=32, N_reg=32, batch_size=65,536
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     32                  | max registers per thread: 255
  Registers per TB:                 1024                
  Shared memory per TB:             4.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     1024                

  Batch size (S):                   65,536              | device memory: 12 GB
  Total TBs in batch:               64.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 2.29):
      registers used per SM:        2,345               | max registers per SM: 65536
      shared memory used per SM:    9.43                | max shared mem per SM: 100 KB
      threads used per SM:          73

In [39]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**17))

Configuration: N_T=32, N_reg=32, batch_size=131,072
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     32                  | max registers per thread: 255
  Registers per TB:                 1024                
  Shared memory per TB:             4.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     1024                

  Batch size (S):                   131,072             | device memory: 12 GB
  Total TBs in batch:               128.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 4.57):
      registers used per SM:        4,680               | max registers per SM: 65536
      shared memory used per SM:    18.83               | max shared mem per SM: 100 KB
      threads used per SM:          1

In [40]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**18))

Configuration: N_T=32, N_reg=32, batch_size=262,144
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     32                  | max registers per thread: 255
  Registers per TB:                 1024                
  Shared memory per TB:             4.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     1024                

  Batch size (S):                   262,144             | device memory: 12 GB
  Total TBs in batch:               256.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 9.14):
      registers used per SM:        9,359               | max registers per SM: 65536
      shared memory used per SM:    37.66               | max shared mem per SM: 100 KB
      threads used per SM:          2

In [41]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**19))

Configuration: N_T=32, N_reg=32, batch_size=524,288
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     32                  | max registers per thread: 255
  Registers per TB:                 1024                
  Shared memory per TB:             4.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     1024                

  Batch size (S):                   524,288             | device memory: 12 GB
  Total TBs in batch:               512.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 16):
      registers used per SM:        16,384              | max registers per SM: 65536
      shared memory used per SM:    65.92               | max shared mem per SM: 100 KB
      threads used per SM:          512

In [42]:
N_T, N_reg = multi_block_size(N_T=32, N_reg=64)

In [43]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**16))

Configuration: N_T=32, N_reg=64, batch_size=65,536
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 2048                
  Shared memory per TB:             8.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     2048                

  Batch size (S):                   65,536              | device memory: 12 GB
  Total TBs in batch:               32.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 1.14):
      registers used per SM:        2,335               | max registers per SM: 65536
      shared memory used per SM:    9.26                | max shared mem per SM: 100 KB
      threads used per SM:          36

In [44]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**17))

Configuration: N_T=32, N_reg=64, batch_size=131,072
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 2048                
  Shared memory per TB:             8.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     2048                

  Batch size (S):                   131,072             | device memory: 12 GB
  Total TBs in batch:               64.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 2.29):
      registers used per SM:        4,690               | max registers per SM: 65536
      shared memory used per SM:    18.59               | max shared mem per SM: 100 KB
      threads used per SM:          7

In [45]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**18))

Configuration: N_T=32, N_reg=64, batch_size=262,144
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 2048                
  Shared memory per TB:             8.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     2048                

  Batch size (S):                   262,144             | device memory: 12 GB
  Total TBs in batch:               128.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 4.57):
      registers used per SM:        9,359               | max registers per SM: 65536
      shared memory used per SM:    37.11               | max shared mem per SM: 100 KB
      threads used per SM:          1

In [46]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**19))

Configuration: N_T=32, N_reg=64, batch_size=524,288
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 2048                
  Shared memory per TB:             8.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     2048                

  Batch size (S):                   524,288             | device memory: 12 GB
  Total TBs in batch:               256.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 9.14):
      registers used per SM:        18,719              | max registers per SM: 65536
      shared memory used per SM:    74.22               | max shared mem per SM: 100 KB
      threads used per SM:          2

In [47]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**20))

Configuration: N_T=32, N_reg=64, batch_size=1,048,576
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 2048                
  Shared memory per TB:             8.12 KB             | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     2048                

  Batch size (S):                   1,048,576           | device memory: 12 GB
  Total TBs in batch:               512.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 12):
      registers used per SM:        24,576              | max registers per SM: 65536
      shared memory used per SM:    97.44               | max shared mem per SM: 100 KB
      threads used per SM:          3

In [48]:
N_T, N_reg = multi_block_size(N_T=32, N_reg=128)

In [49]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**16))

Configuration: N_T=32, N_reg=128, batch_size=65,536
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     128                 | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.12 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   65,536              | device memory: 12 GB
  Total TBs in batch:               16.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 0.57):
      registers used per SM:        2,335               | max registers per SM: 65536
      shared memory used per SM:    9.19                | max shared mem per SM: 100 KB
      threads used per SM:          1

In [50]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**17))

Configuration: N_T=32, N_reg=128, batch_size=131,072
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     128                 | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.12 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   131,072             | device memory: 12 GB
  Total TBs in batch:               32.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 1.14):
      registers used per SM:        4,669               | max registers per SM: 65536
      shared memory used per SM:    18.38               | max shared mem per SM: 100 KB
      threads used per SM:          

In [51]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**18))

Configuration: N_T=32, N_reg=128, batch_size=262,144
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     128                 | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.12 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   262,144             | device memory: 12 GB
  Total TBs in batch:               64.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 2.29):
      registers used per SM:        9,380               | max registers per SM: 65536
      shared memory used per SM:    36.91               | max shared mem per SM: 100 KB
      threads used per SM:          

In [52]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**19))

Configuration: N_T=32, N_reg=128, batch_size=524,288
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     128                 | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.12 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   524,288             | device memory: 12 GB
  Total TBs in batch:               128.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 4.57):
      registers used per SM:        18,719              | max registers per SM: 65536
      shared memory used per SM:    73.67               | max shared mem per SM: 100 KB
      threads used per SM:          

In [53]:
N_T, N_reg = multi_block_size(N_T=64, N_reg=64)

In [54]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**16))

Configuration: N_T=64, N_reg=64, batch_size=65,536
-----------------------------------------------------------------------------------------------
  Threads per TB:                   64                  | max threads per TB: 1024
  Warps per TB:                     2                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.25 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   65,536              | device memory: 12 GB
  Total TBs in batch:               16.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 0.57):
      registers used per SM:        2,335               | max registers per SM: 65536
      shared memory used per SM:    9.26                | max shared mem per SM: 100 KB
      threads used per SM:          36

In [55]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**17))

Configuration: N_T=64, N_reg=64, batch_size=131,072
-----------------------------------------------------------------------------------------------
  Threads per TB:                   64                  | max threads per TB: 1024
  Warps per TB:                     2                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.25 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   131,072             | device memory: 12 GB
  Total TBs in batch:               32.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 1.14):
      registers used per SM:        4,669               | max registers per SM: 65536
      shared memory used per SM:    18.52               | max shared mem per SM: 100 KB
      threads used per SM:          7

In [56]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**18))

Configuration: N_T=64, N_reg=64, batch_size=262,144
-----------------------------------------------------------------------------------------------
  Threads per TB:                   64                  | max threads per TB: 1024
  Warps per TB:                     2                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.25 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   262,144             | device memory: 12 GB
  Total TBs in batch:               64.0                | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 2.29):
      registers used per SM:        9,380               | max registers per SM: 65536
      shared memory used per SM:    37.21               | max shared mem per SM: 100 KB
      threads used per SM:          1

In [57]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**19))

Configuration: N_T=64, N_reg=64, batch_size=524,288
-----------------------------------------------------------------------------------------------
  Threads per TB:                   64                  | max threads per TB: 1024
  Warps per TB:                     2                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.25 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   524,288             | device memory: 12 GB
  Total TBs in batch:               128.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 4.57):
      registers used per SM:        18,719              | max registers per SM: 65536
      shared memory used per SM:    74.26               | max shared mem per SM: 100 KB
      threads used per SM:          2

In [58]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**20))

Configuration: N_T=64, N_reg=64, batch_size=1,048,576
-----------------------------------------------------------------------------------------------
  Threads per TB:                   64                  | max threads per TB: 1024
  Warps per TB:                     2                   
  Registers per thread (N_reg):     64                  | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.25 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   1,048,576           | device memory: 12 GB
  Total TBs in batch:               256.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 6):
      registers used per SM:        24,576              | max registers per SM: 65536
      shared memory used per SM:    97.50               | max shared mem per SM: 100 KB
      threads used per SM:          38

In [59]:
N_T, N_reg = multi_block_size(N_T=32, N_reg=128)

In [60]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**19))

Configuration: N_T=32, N_reg=128, batch_size=524,288
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     128                 | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.12 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   524,288             | device memory: 12 GB
  Total TBs in batch:               128.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 4.57):
      registers used per SM:        18,719              | max registers per SM: 65536
      shared memory used per SM:    73.67               | max shared mem per SM: 100 KB
      threads used per SM:          

In [62]:
print_occupancy(compute_occupancy(N_T=N_T, N_reg=N_reg,batch_size=2**20))

Configuration: N_T=32, N_reg=128, batch_size=1,048,576
-----------------------------------------------------------------------------------------------
  Threads per TB:                   32                  | max threads per TB: 1024
  Warps per TB:                     1                   
  Registers per thread (N_reg):     128                 | max registers per thread: 255
  Registers per TB:                 4096                
  Shared memory per TB:             16.12 KB            | max shared mem per TB: 48 KB
  Samples per TB (N_reg * N_T):     4096                

  Batch size (S):                   1,048,576           | device memory: 12 GB
  Total TBs in batch:               256.0               | num SMs: 28

  Per-SM utilization at effective residency (TBs/SM = 6):
      registers used per SM:        24,576              | max registers per SM: 65536
      shared memory used per SM:    96.72               | max shared mem per SM: 100 KB
      threads used per SM:          1